# 02 — Building Neural Networks

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/06_Deep_Learning/02_building_neural_networks.ipynb)

## 📌 What is it?
PyTorch's `nn.Module` is the base class for all neural networks. You define the architecture in `__init__` and the forward pass in `forward()`.

## 🧠 Why AI Engineers need this
Custom model heads, LoRA adapters, domain-specific architectures, and classifier layers all require writing `nn.Module` subclasses.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# ── BUILD A FEEDFORWARD NETWORK ──
class FeedForwardNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    
    def forward(self, x):
        return self.network(x)

model = FeedForwardNet(input_dim=128, hidden_dim=256, output_dim=2)
print(model)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"
Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")

# Test forward pass
x = torch.randn(32, 128)  # batch of 32 samples
out = model(x)
print(f"
Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ── FULL TRAINING LOOP ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data
torch.manual_seed(42)
X = torch.randn(500, 128)
y = (X[:, 0] + X[:, 1] > 0).long()

dataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)

# Model, loss, optimizer
model     = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 2)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# Training loop
for epoch in range(10):
    model.train()
    epoch_loss, correct = 0.0, 0
    
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()          # 1. clear gradients
        logits = model(xb)             # 2. forward pass
        loss = criterion(logits, yb)   # 3. compute loss
        loss.backward()                # 4. backward pass
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # clip grads
        optimizer.step()               # 5. update weights
        
        epoch_loss += loss.item()
        correct += (logits.argmax(1) == yb).sum().item()
    
    scheduler.step()
    acc = correct / len(dataset)
    if epoch % 2 == 0:
        print(f"Epoch {epoch:2d} | Loss: {epoch_loss/len(loader):.4f} | Acc: {acc:.3f}")

In [ ]:
import torch
import torch.nn as nn

# ── SAVE AND LOAD MODEL ──
model = nn.Linear(10, 2)

# Save (weights only — recommended)
torch.save(model.state_dict(), "/tmp/model_weights.pt")

# Load
loaded = nn.Linear(10, 2)
loaded.load_state_dict(torch.load("/tmp/model_weights.pt", map_location="cpu"))
loaded.eval()  # switch to inference mode

print("✅ Model saved and loaded successfully")
print(f"Weights match: {all(torch.equal(a, b) for a, b in zip(model.parameters(), loaded.parameters()))}")

## 🔗 What's Next?
[03 — Keras & TensorFlow →](03_keras_and_tensorflow.ipynb)